# Create a theme vote (test)

Builds a `ThemeVote` with `ThemeVoteGenerator` (selection + schedule
strategies, no database) and persists it with `add_theme_vote`, so the voting
flow can be exercised without posting to Bluesky. This notebook only touches
the database — no posts or replies are made.

In [1]:
import datetime
import os
import sys

# allow importing the project's packages when run from tools/
sys.path.insert(0, os.path.abspath('..'))

from core.database import session_scope, init_db
from core.interactions.votes import (
    add_theme_vote,
    get_theme_vote,
    all_theme_votes,
    delete_theme_vote,
)
from core.interactions.votes import ThemeVoteGenerator

# make sure the theme_votes / theme_options tables exist
init_db()

Build the vote (no database needed). Defaults give a random handful of themes
and the default schedule from `now()`; pass a `start_date`, or custom
selection / schedule strategies, to mix and match.

In [2]:
vote = ThemeVoteGenerator().generate()

# e.g. an explicit start date and a fixed ballot size:
# from core.interactions.votes import AllThemesSelection
# vote = ThemeVoteGenerator(AllThemesSelection(count=4)).generate(datetime.datetime(2026, 7, 1))

print('built (unsaved) vote — id:', vote.id)
print(f'  type:         {vote.type.value}')
print(f'  voting:       {vote.vote_start_date} -> {vote.vote_end_date}')
print(f'  theme window: {vote.theme_start_date} -> {vote.theme_end_date}')
print('  options:', [o.theme_tag for o in vote.options])

built (unsaved) vote — id: None
  type:         default
  voting:       2026-06-13 23:23:23.344181 -> 2026-06-14 23:23:23.344181
  theme window: 2026-06-14 23:23:23.344181 -> 2026-06-16 23:23:23.344181
  options: ['garfield', 'rocks', 'sandstone', 'sand', 'winter']


In [3]:
with session_scope() as session:
    vote = add_theme_vote(session, vote)

print(f'saved vote {vote.id}')
for option in vote.options:
    print(f'    - {option.theme_name} ({option.theme_tag})  [option {option.id}]')

saved vote 1
    - Garfield (garfield)  [option 1]
    - Rocks (rocks)  [option 2]
    - Sandstone (sandstone)  [option 3]
    - Sand (sand)  [option 4]
    - Winter (winter)  [option 5]


Read the vote back out of the database (this is the shape the posting /
tallying steps will consume). `comment_uri` / `comment_cid` and `likes` stay
empty until the Bluesky replies are made and checked.

In [4]:
with session_scope() as session:
    loaded = get_theme_vote(session, vote.id)

loaded

ThemeVote(id=1, type=<ThemeVoteType.DEFAULT: 'default'>, post_uri=None, post_cid=None, vote_start_date=datetime.datetime(2026, 6, 13, 23, 23, 23, 344181), vote_end_date=datetime.datetime(2026, 6, 14, 23, 23, 23, 344181), theme_start_date=datetime.datetime(2026, 6, 14, 23, 23, 23, 344181), theme_end_date=datetime.datetime(2026, 6, 16, 23, 23, 23, 344181), options=[ThemeOption(id=1, theme_tag='garfield', theme_name='Garfield', comment_uri=None, comment_cid=None, likes=None), ThemeOption(id=2, theme_tag='rocks', theme_name='Rocks', comment_uri=None, comment_cid=None, likes=None), ThemeOption(id=3, theme_tag='sandstone', theme_name='Sandstone', comment_uri=None, comment_cid=None, likes=None), ThemeOption(id=4, theme_tag='sand', theme_name='Sand', comment_uri=None, comment_cid=None, likes=None), ThemeOption(id=5, theme_tag='winter', theme_name='Winter', comment_uri=None, comment_cid=None, likes=None)])

In [ ]:
# all votes currently stored
with session_scope() as session:
    votes = all_theme_votes(session)

for v in votes:
    print(f'{v.id}  {v.type.value}  ({len(v.options)} options)')

### Clean up test data

Uncomment to delete the vote created above (also removes its options).

In [ ]:
# with session_scope() as session:
#     removed = delete_theme_vote(session, vote.id)
# print('deleted' if removed else 'nothing to delete')